# Insurance Pricing — Monte Carlo Simulation

This notebook applies **Monte Carlo simulation** to the insurance pricing factor models.  
By running thousands of trials, we can:

- Estimate the **expected premium** for each model version
- Quantify **uncertainty** and risk ranges
- Compare model stability as complexity increases
- Compute **Value at Risk (VaR)** and **Conditional Tail Expectation (CTE)** for each model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Configuration

In [ ]:
N_SIMULATIONS = 10_000
BASE_PREMIUM = 500  # base annual premium in GBP

factor_counts = [2, 3, 4, 5, 6, 7, 8, 9]
model_names = [f"Model v{i} ({c} factors)" for i, c in enumerate(factor_counts)]

np.random.seed(42)

## 2. Monte Carlo Premium Simulation

For each model version, we simulate `N_SIMULATIONS` premium calculations.  
The premium is computed as:

$$\text{Premium} = \text{Base} \times \prod_{j=1}^{k} (0.5 + f_j)$$

where \(f_j \sim \text{Uniform}(0, 1)\) are the rating factors and the \(0.5\) offset ensures the multiplier stays in \([0.5, 1.5]\).

In [ ]:
simulation_results = {}

for i, count in enumerate(factor_counts):
    factors = np.random.uniform(0, 1, size=(N_SIMULATIONS, count))
    multipliers = 0.5 + factors  # shift to [0.5, 1.5]
    premiums = BASE_PREMIUM * np.prod(multipliers, axis=1)
    simulation_results[model_names[i]] = premiums

print(f"Simulated {N_SIMULATIONS} premiums for {len(factor_counts)} model versions.")

## 3. Summary Statistics

In [ ]:
summary = []
for name, premiums in simulation_results.items():
    summary.append({
        "Model": name,
        "Mean Premium (£)": np.round(premiums.mean(), 2),
        "Median (£)": np.round(np.median(premiums), 2),
        "Std Dev (£)": np.round(premiums.std(), 2),
        "Min (£)": np.round(premiums.min(), 2),
        "Max (£)": np.round(premiums.max(), 2),
        "VaR 95% (£)": np.round(np.percentile(premiums, 95), 2),
        "CTE 95% (£)": np.round(premiums[premiums >= np.percentile(premiums, 95)].mean(), 2)
    })

df_summary = pd.DataFrame(summary).set_index("Model")
df_summary

## 4. Premium Distribution per Model

Overlaid histograms showing how premium distributions widen with more factors.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=False, sharey=False)
axes = axes.flatten()

colors = plt.cm.tab10(np.linspace(0, 0.8, len(factor_counts)))

for idx, (name, premiums) in enumerate(simulation_results.items()):
    ax = axes[idx]
    ax.hist(premiums, bins=60, color=colors[idx], edgecolor='white', alpha=0.85)
    ax.axvline(premiums.mean(), color='red', linestyle='--', linewidth=1.2)
    ax.axvline(np.percentile(premiums, 95), color='darkred', linestyle=':', linewidth=1.2)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel("Premium (£)", fontsize=8)
    ax.tick_params(labelsize=8)

fig.suptitle("Simulated Premium Distributions by Model Version", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Box Plot Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

data_for_box = [premiums for premiums in simulation_results.values()]
bp = ax.boxplot(data_for_box, patch_artist=True, labels=[f"v{i}\n({c}f)" for i, c in enumerate(factor_counts)])

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel("Simulated Premium (£)", fontsize=11)
ax.set_xlabel("Model Version (number of factors)", fontsize=11)
ax.set_title("Premium Spread Across Model Versions (10,000 simulations)", fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Convergence Analysis

Track how the running mean premium stabilises as the number of simulations increases.  
This validates whether 10,000 simulations is sufficient.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sample_models = [0, 3, 7]  # v0 (2f), v3 (5f), v7 (9f)

for idx in sample_models:
    name = model_names[idx]
    premiums = simulation_results[name]
    running_mean = np.cumsum(premiums) / np.arange(1, len(premiums) + 1)
    ax.plot(running_mean, label=name, linewidth=1.2)

ax.set_xlabel("Number of Simulations", fontsize=11)
ax.set_ylabel("Running Mean Premium (£)", fontsize=11)
ax.set_title("Convergence of Mean Premium Estimate", fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Risk Metrics — VaR & CTE by Model Complexity

In [ ]:
var_95 = df_summary["VaR 95% (£)"].values
cte_95 = df_summary["CTE 95% (£)"].values
x = np.arange(len(factor_counts))

fig, ax = plt.subplots(figsize=(11, 5))
w = 0.35
ax.bar(x - w/2, var_95, w, label="VaR 95%", color='steelblue', edgecolor='white')
ax.bar(x + w/2, cte_95, w, label="CTE 95%", color='coral', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([f"v{i} ({c}f)" for i, c in enumerate(factor_counts)], fontsize=9)
ax.set_ylabel("Premium (£)", fontsize=11)
ax.set_title("Value at Risk & Conditional Tail Expectation (95th percentile)", fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Key Findings

- As the number of pricing factors increases, the **premium distribution widens** significantly
- Models with more factors have higher **tail risk** (VaR and CTE increase)
- The multiplicative premium formula causes the mean premium to grow with model complexity
- 10,000 simulations provide stable mean estimates (convergence plot confirms this)